# revT2V: final evaluation (Colab, GPU)

Reuses the setup and helper patterns from `notebooks/compare_methods.ipynb`
and the noising loop in `scripts/train.py` -- no logic is duplicated, only
re-imported. Sections, each saving into `results/final/`:

1. Exactness checks for `conv_oracle`.
2. Generation for every available method from the same initial noise.
3. Metrics per method vs the matched target.
4. Diagnostic rho for the trained students + `scripts/conv_rank_check.py`.
5. Zip `results/final/` for download.

Set `QUICK` below: `True` runs 5 prompts x 1 seed (~15 min on a T4);
`False` runs every `prompts/test.txt` prompt x 2 seeds (the full report run).

In [ ]:
# 1. Setup -- clone/pull, install, GPU check. No %autoreload: this notebook
# is run top-to-bottom on a fresh Colab runtime, not iterated on in place.
import os
import subprocess

REPO_URL = "https://github.com/silentlooop/revT2V.git"
REPO_DIR = "/content/revT2V"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
os.chdir(REPO_DIR)

subprocess.run(["pip", "install", "-q", "-e", ".[eval]"], check=True)

# HF_TOKEN from a Colab secret (left sidebar, key icon) -- never hardcode it.
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception as exc:
    raise RuntimeError(
        "Add a Colab secret named HF_TOKEN with read access to "
        "silentlooop/revt2v-ckpt and silentlooop/revt2v-data, and turn on "
        "notebook access for it."
    ) from exc

import torch
print("CUDA:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU -- stop and switch runtime type")

In [ ]:
QUICK = True  # True: 5 prompts x 1 seed (~15 min). False: full prompts/test.txt x 2 seeds.

HF_REPO = "silentlooop/revt2v-ckpt"
OUT_DIR = "results/final"
VIDEOS_DIR = f"{OUT_DIR}/videos"
os.makedirs(OUT_DIR, exist_ok=True)

with open("prompts/test.txt") as f:
    TEST_PROMPTS = [line.strip() for line in f if line.strip()]

PROMPTS = TEST_PROMPTS[:5] if QUICK else TEST_PROMPTS
SEEDS = [0] if QUICK else [0, 1]

GEN = dict(
    negative_prompt="watermark, text",
    num_inference_steps=25,
    guidance_scale=9.0,
    num_frames=16,
    height=256,
    width=256,
)
FPS = 8
print(f"QUICK={QUICK} | {len(PROMPTS)} prompts x {len(SEEDS)} seed(s) = {len(PROMPTS) * len(SEEDS)} clips per method")

In [ ]:
# Load the teacher once, then every trained adapter that has a checkpoint.
# A missing checkpoint is skipped with a message here, not a crash -- see
# revt2v.infer.MethodBank.load_adapter.
from revt2v.infer import MethodBank

bank = MethodBank(hf_repo_id=HF_REPO)
bank.pipeline.set_progress_bar_config(disable=True)

for method in ["conv_lora", "attn_lora", "attn_injection"]:
    try:
        loaded = bank.load_adapter(method)
        print(method, "loaded" if loaded else "-> no checkpoint found, skipped")
    except Exception as exc:  # e.g. a checkpoint saved by a different method
        print(method, "-> skipped:", exc)

print("available methods:", bank.methods)

## 1. Exactness checks for `conv_oracle`

In [ ]:
# 1a. U-Net level: unet_flipped(z, t, emb) vs flip(unet(flip(z), t, emb)).
# Same check as tests/test_conv_mirror.py's unit test, run here against the
# real loaded pipeline/weights instead of a tiny synthetic U-Net.
from contextlib import nullcontext

from revt2v.methods.conv_mirror import find_temporal_convs, flip_temporal_convs, set_flip_state
from revt2v.teacher import encode_prompt

unet = bank.pipeline.unet
print(len(find_temporal_convs(unet)), "temporal Conv3d layers")


def initial_noise(seed):
    g = torch.Generator(device="cuda").manual_seed(seed)
    return torch.randn(
        (1, 4, GEN["num_frames"], GEN["height"] // 8, GEN["width"] // 8),
        generator=g, device="cuda", dtype=unet.dtype,
    )


def to_np(video):
    return video.float().clamp(0, 1).cpu().numpy()


z = initial_noise(0)
emb, _ = encode_prompt(bank.pipeline, PROMPTS[0])
emb = emb.to(unet.dtype)
t = torch.tensor([500], device="cuda")

adapters_off = bank.student.disable_adapter() if bank.student is not None else nullcontext()
unet_exactness_report = {}
with adapters_off:
    with torch.no_grad():
        mirrored = unet(z.flip(2), t, encoder_hidden_states=emb, return_dict=False)[0].flip(2)
        flip_temporal_convs(unet)
        try:
            flipped = unet(z, t, encoder_hidden_states=emb, return_dict=False)[0]
        finally:
            set_flip_state(unet, ())  # always undo the flip, even if the forward pass raised
    max_diff = (mirrored - flipped).abs().max().item()
unet_exactness_report["unet_level_max_abs_diff"] = max_diff
print(f"U-Net level: max |mirrored - flipped| = {max_diff:.3e}")

In [ ]:
# 1b. Video level: conv_oracle(latents=z) vs matched_target(prompt, z).
z = initial_noise(0)
oracle = to_np(bank.generate(PROMPTS[0], "conv_oracle", latents=z, **GEN)["video"])
target = to_np(bank.matched_target(PROMPTS[0], z, **GEN)["video"])
mean_pixel_diff = abs(oracle - target).mean()
unet_exactness_report["video_level_mean_pixel_diff"] = float(mean_pixel_diff)
print(f"Video level: mean pixel diff (0-1 scale) = {mean_pixel_diff:.4f}")

import json as _json
with open(f"{OUT_DIR}/exactness_report.json", "w") as f:
    _json.dump(unet_exactness_report, f, indent=2)
print(f"wrote {OUT_DIR}/exactness_report.json")

## 2. Generation: every available method from the same noise

In [ ]:
import sys

import numpy as np
from diffusers.utils import export_to_video

# scripts/ isn't a package (and `evaluate` would clash with Hugging Face's
# `evaluate` library) -- same sys.path trick server/app.py and
# compare_methods.ipynb use to import it.
sys.path.insert(0, "scripts")
import evaluate as ev

COLUMNS = [m for m in ["teacher", "matched_target", "attn_injection", "attn_lora", "conv_oracle", "conv_lora"]
           if m == "matched_target" or m in bank.methods]
print("comparison columns:", COLUMNS)

videos, names = ev.generate_videos(bank, PROMPTS, SEEDS, [m for m in COLUMNS if m != "matched_target"], GEN)
ev.save_video_set(videos, names, VIDEOS_DIR, fps=FPS)

os.makedirs(f"{OUT_DIR}/side_by_side", exist_ok=True)
for name, prompt in names.items():
    side_by_side = np.clip(np.concatenate([videos[m][name] for m in COLUMNS], axis=2), 0, 1)
    export_to_video(list(side_by_side), f"{OUT_DIR}/side_by_side/{name}.mp4", fps=FPS)
    with open(f"{OUT_DIR}/side_by_side/{name}.json", "w") as f:
        _json.dump({"columns": COLUMNS, "prompt": prompt}, f, indent=2)
print(f"wrote {len(names)} side-by-side comparison video(s) to {OUT_DIR}/side_by_side/ (columns recorded per-file)")

## 3. Metrics per method vs the matched target

In [ ]:
import pandas as pd

# First run downloads CLIP ViT-L/14, I3D and the VBench-style models (a few GB).
# Writes metrics.csv, summary.csv and summary.json into OUT_DIR.
result = ev.score_videos_dir(VIDEOS_DIR, output_dir=OUT_DIR)

summary = pd.DataFrame(result["summary"]).set_index("method")
display(summary.round(3))
display(pd.DataFrame(result["rows"]).round(3))
print(f"wrote {OUT_DIR}/metrics.csv, {OUT_DIR}/summary.csv, {OUT_DIR}/summary.json")

## 4. Diagnostic: fixed-batch rho for the trained students

    rho = ||eps_student - eps_oracle||^2 / ||eps_teacher - eps_oracle||^2
    eps_oracle = flip(eps_teacher(flip(x_t)))

on `N_FIXED_CLIPS` fixed reversed clips x `FIXED_TIMESTEPS`, with fixed noise.
rho=1: no better than the plain forward teacher. rho=0: perfect match to the
oracle. This mirrors `scripts/train.py`'s own noising loop exactly (same
`DDPMScheduler.from_config`, same `add_noise`, same `Student.predict_noise`)
so the numbers are directly comparable to training-time losses -- eval-only,
no gradients, no backward.

In [ ]:
import csv

from diffusers import DDPMScheduler
import yaml

from revt2v.data import LatentDataset
from revt2v.methods import METHODS
from revt2v.methods.conv_mirror import mirrored_teacher_noise
from revt2v.student import Student

FIXED_TIMESTEPS = [300, 500, 700, 900]
N_FIXED_CLIPS = 16
FIXED_NOISE_SEED = 0

noise_scheduler = DDPMScheduler.from_config(bank.pipeline.scheduler.config)

# Any config's data_root works -- conv_lora/attn_lora/attn_injection all
# train off the same dataset (see configs/*.yaml).
with open("configs/conv_lora.yaml") as f:
    data_root = yaml.safe_load(f)["data_root"]
dataset = LatentDataset(data_root)
n_fixed = min(N_FIXED_CLIPS, len(dataset))
if n_fixed < N_FIXED_CLIPS:
    print(f"only {len(dataset)} clips available, using all {n_fixed} (wanted {N_FIXED_CLIPS})")
fixed_records = [dataset[i] for i in range(n_fixed)]

reversed_latents = torch.stack([r["reversed_latents"] for r in fixed_records]).to("cuda", unet.dtype)
prompt_embeds = torch.stack([r["prompt_embeds"] for r in fixed_records]).to("cuda", unet.dtype)

g = torch.Generator(device="cuda").manual_seed(FIXED_NOISE_SEED)
fixed_noise = torch.randn(reversed_latents.shape, generator=g, device="cuda", dtype=reversed_latents.dtype)

rho_rows = []
trained_methods = [m for m in bank.methods if m not in ("teacher", "conv_oracle")]
if not trained_methods:
    print("no trained-student checkpoints loaded -- nothing to diagnose")

for method_name in trained_methods:
    bank.load_adapter(method_name)
    student = bank.student
    method = METHODS[method_name]()
    for t_val in FIXED_TIMESTEPS:
        timesteps = torch.full((reversed_latents.shape[0],), t_val, device="cuda", dtype=torch.long)
        noisy_latents = noise_scheduler.add_noise(reversed_latents, fixed_noise, timesteps)

        # Mirrors scripts/train.py's loop: attn_injection needs prepare_step
        # to refresh its captured attention maps before predict_noise reads
        # them; every other method's prepare_step is a no-op.
        getattr(method, "prepare_step", lambda *a, **k: None)(student, noisy_latents, timesteps, prompt_embeds)

        with torch.no_grad():
            eps_oracle = mirrored_teacher_noise(student, noisy_latents, timesteps, prompt_embeds)
            with student.disable_adapter():
                eps_teacher = unet(noisy_latents, timesteps, encoder_hidden_states=prompt_embeds, return_dict=False)[0]
            eps_student = Student.predict_noise(student, noisy_latents, timesteps, prompt_embeds)

        se_student = (eps_student - eps_oracle).pow(2).sum().item()
        se_teacher = (eps_teacher - eps_oracle).pow(2).sum().item()
        rho = se_student / se_teacher if se_teacher else float("nan")
        rho_rows.append({"method": method_name, "t": t_val, "rho": rho})
        print(f"{method_name:>15s}  t={t_val:>4d}  rho={rho:.4f}")

if rho_rows:
    with open(f"{OUT_DIR}/rho_diagnostic.csv", "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["method", "t", "rho"])
        writer.writeheader()
        writer.writerows(rho_rows)
    print(f"wrote {OUT_DIR}/rho_diagnostic.csv")

In [ ]:
# scripts/conv_rank_check.py's own summary (energy-fraction rank recommendation).
rank_check = subprocess.run(
    [sys.executable, "scripts/conv_rank_check.py"], capture_output=True, text=True, check=True
)
print(rank_check.stdout)
with open(f"{OUT_DIR}/rank_check.txt", "w") as f:
    f.write(rank_check.stdout)
print(f"wrote {OUT_DIR}/rank_check.txt")

## 5. Zip `results/final/` for download

In [ ]:
import shutil

zip_path = shutil.make_archive("results/final", "zip", root_dir=OUT_DIR)
size_mb = os.path.getsize(zip_path) / (1024 * 1024)
print(f"wrote {zip_path} ({size_mb:.1f} MB)")

try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    print("not running in Colab (or download blocked) -- fetch results/final.zip manually")